In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04 – Results Visualisation\n",
    "\n",
    "This notebook visualises model performance using:\n",
    "\n",
    "- Confusion matrix\n",
    "- Precision–recall curves (one-vs-rest) for each posture class\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import StandardScaler, LabelEncoder\n",
    "from sklearn.metrics import (\n",
    "    confusion_matrix,\n",
    "    ConfusionMatrixDisplay,\n",
    "    precision_recall_curve,\n",
    ")\n",
    "\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "import numpy as np\n",
    "\n",
    "# Load dataset\n",
    "df = pd.read_csv(\"../data/sample_posture_data.csv\")\n",
    "\n",
    "X = df.drop(columns=[\"label\"])\n",
    "y = df[\"label\"]\n",
    "\n",
    "encoder = LabelEncoder()\n",
    "y_enc = encoder.fit_transform(y)\n",
    "\n",
    "scaler = StandardScaler()\n",
    "X_scaled = scaler.fit_transform(X)\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X_scaled,\n",
    "    y_enc,\n",
    "    test_size=0.2,\n",
    "    random_state=42,\n",
    "    stratify=y_enc,\n",
    ")\n",
    "\n",
    "# Train reference model (Random Forest)\n",
    "model = RandomForestClassifier(n_estimators=200, random_state=42)\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "y_pred = model.predict(X_test)\n",
    "y_proba = model.predict_proba(X_test)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Confusion matrix\n",
    "cm = confusion_matrix(y_test, y_pred)\n",
    "disp = ConfusionMatrixDisplay(cm, display_labels=encoder.classes_)\n",
    "disp.plot(cmap=\"Blues\")\n",
    "plt.title(\"Confusion Matrix – Random Forest\")\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Precision–Recall curves (one-vs-rest)\n",
    "from itertools import cycle\n",
    "\n",
    "n_classes = len(encoder.classes_)\n",
    "colors = cycle([\"b\", \"g\", \"r\", \"c\", \"m\", \"y\", \"k\"])\n",
    "\n",
    "plt.figure()\n",
    "for i, color in zip(range(n_classes), colors):\n",
    "    # One-vs-rest for class i\n",
    "    y_true_binary = (y_test == i).astype(int)\n",
    "    precision, recall, _ = precision_recall_curve(\n",
    "        y_true_binary, y_proba[:, i]\n",
    "    )\n",
    "    plt.plot(\n",
    "        recall,\n",
    "        precision,\n",
    "        color=color,\n",
    "        label=f\"Class {encoder.classes_[i]}\",\n",
    "    )\n",
    "\n",
    "plt.xlabel(\"Recall\")\n",
    "plt.ylabel(\"Precision\")\n",
    "plt.title(\"Precision–Recall Curves (One-vs-Rest)\")\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}